# Salary Linear Regression Baseline

Notebook nay tao baseline Linear Regression cho muc luong IT va minh hoa demand trend theo tuan. Model chinh chi dung `sklearn.linear_model.LinearRegression`.

## 0. Setup

Input chinh:
- `data/analysis/salary_analysis_clean.csv`
- `data/analysis/jobs_analysis_clean.csv`

Output:
- `data/analysis/modeling/salary_linear_regression_predictions.csv`
- `data/analysis/modeling/salary_linear_regression_metrics.csv`
- `data/analysis/modeling/demand_linear_regression_baseline.csv`

In [ ]:
from __future__ import annotations

import ast
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

if "get_ipython" not in globals():
    import matplotlib
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

USD_TO_VND = 26_000
TOP_SKILL_COUNT = 25
RANDOM_STATE = 42


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        salary_path = candidate / "data" / "analysis" / "salary_analysis_clean.csv"
        jobs_path = candidate / "data" / "analysis" / "jobs_analysis_clean.csv"
        if salary_path.exists() and jobs_path.exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing data/analysis inputs")


REPO_ROOT = find_repo_root()
ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"
MODELING_DIR = ANALYSIS_DIR / "modeling"
SALARY_PATH = ANALYSIS_DIR / "salary_analysis_clean.csv"
JOBS_PATH = ANALYSIS_DIR / "jobs_analysis_clean.csv"

MODELING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Salary input: {SALARY_PATH}")
print(f"Jobs input: {JOBS_PATH}")
print(f"Modeling output dir: {MODELING_DIR}")

## 1. Load Data

`salary_analysis_clean.csv` la tap numeric salary only. `jobs_analysis_clean.csv` giu day du source hon de xem demand theo tuan.

In [ ]:
salary_raw = pd.read_csv(SALARY_PATH, encoding="utf-8-sig")
jobs_raw = pd.read_csv(JOBS_PATH, encoding="utf-8-sig")

print(f"Salary rows: {len(salary_raw):,}; unique URLs: {salary_raw['url'].nunique():,}")
print(f"Jobs rows: {len(jobs_raw):,}; unique URLs: {jobs_raw['url'].nunique():,}")

assert len(salary_raw) > 0, "salary_analysis_clean.csv must contain rows"
assert len(jobs_raw) > 0, "jobs_analysis_clean.csv must contain rows"
assert salary_raw["url"].is_unique, "salary dataset should be deduped by url"

display(salary_raw[["source", "title", "company", "location", "salary_raw", "salary_currency", "salary_period", "salary_midpoint", "seniority", "experience_min"]].head(10))
display(jobs_raw[["source", "title", "company", "location", "posted_at", "skills"]].head(10))

## 2. Helpers

Cac helper nay tao lai feature tu notebook visualization: monthly VND salary, log salary, location normalized, skills parsed, skill_count, posted_age_days.

In [ ]:
def strip_accents(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).translate({ord("\u0110"): "D", ord("\u0111"): "d"})
    return "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )


def fold_text(value) -> str:
    folded = strip_accents(value).lower()
    return re.sub(r"\s+", " ", folded).strip()


def clean_text_series(series: pd.Series) -> pd.Series:
    return series.astype("string").str.replace(r"\s+", " ", regex=True).str.strip()


def normalize_location(value) -> str:
    folded = fold_text(value)
    if re.search(r"\b(?:ho chi minh|hcm|sai gon)\b", folded):
        return "Ho Chi Minh"
    if re.search(r"\b(?:ha noi|hanoi)\b", folded):
        return "Ha Noi"
    if re.search(r"\b(?:da nang|danang)\b", folded):
        return "Da Nang"
    if "remote" in folded:
        return "Remote"
    if not folded or folded in {"nan", "none", "null"}:
        return "Unknown"
    return str(value).strip()


def to_listish(value) -> list[str]:
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (ValueError, SyntaxError):
            pass
    return [part.strip() for part in text.replace(";", ",").split(",") if part.strip()]


SKILL_ALIASES = {
    "js": "javascript",
    "reactjs": "react",
    "react.js": "react",
    "vuejs": "vue",
    "vue.js": "vue",
    "node": "node.js",
    "nodejs": "node.js",
    "golang": "go",
    "postgres": "postgresql",
    "k8s": "kubernetes",
    "ts": "typescript",
    "csharp": "c#",
    "dotnet": ".net",
    "html5": "html",
    "css3": "css",
}


def normalize_skill(value) -> str:
    folded = fold_text(value).replace("&", " and ")
    folded = re.sub(r"\s+", " ", folded).strip()
    return SKILL_ALIASES.get(folded, folded)


def prepare_salary_model_frame(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.copy()

    text_columns = ["source", "title", "company", "location", "salary_raw", "salary_currency", "salary_period", "seniority", "work_mode", "employment_type", "skills"]
    for column in text_columns:
        if column in frame.columns:
            frame[column] = clean_text_series(frame[column])

    for column in ["salary_min", "salary_max", "salary_midpoint", "experience_min", "experience_max"]:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["scraped_at_dt"] = pd.to_datetime(frame["scraped_at"], errors="coerce", utc=True, format="mixed")
    frame["posted_at_dt"] = pd.to_datetime(frame["posted_at"], errors="coerce", utc=True, format="mixed")
    frame["posted_age_days"] = (frame["scraped_at_dt"] - frame["posted_at_dt"]).dt.days

    salary_raw_folded = frame["salary_raw"].map(fold_text)
    raw_has_annual_signal = salary_raw_folded.str.contains(r"\b(?:year|annual|annually|nam)\b", na=False)
    frame["salary_period_clean"] = frame["salary_period"].astype("string")
    frame.loc[frame["salary_period_clean"].eq("year") & ~raw_has_annual_signal, "salary_period_clean"] = "month"

    frame["salary_monthly_vnd"] = frame["salary_midpoint"]
    usd_mask = frame["salary_currency"].astype("string").str.upper().eq("USD")
    annual_mask = frame["salary_period_clean"].eq("year") & raw_has_annual_signal
    frame.loc[usd_mask, "salary_monthly_vnd"] = frame.loc[usd_mask, "salary_monthly_vnd"] * USD_TO_VND
    frame.loc[annual_mask, "salary_monthly_vnd"] = frame.loc[annual_mask, "salary_monthly_vnd"] / 12
    frame["salary_monthly_vnd_m"] = frame["salary_monthly_vnd"] / 1_000_000
    frame["log_salary_monthly_vnd"] = np.log(frame["salary_monthly_vnd"].where(frame["salary_monthly_vnd"].gt(0)))

    frame["location_norm"] = frame["location"].map(normalize_location).astype("string")
    frame["skills_list"] = frame["skills"].apply(to_listish)
    frame["skills_norm_list"] = frame["skills_list"].apply(
        lambda values: sorted({normalize_skill(value) for value in values if normalize_skill(value)})
    )
    frame["skill_count"] = frame["skills_norm_list"].str.len()

    return frame


def add_skill_flags(frame: pd.DataFrame, top_skills: list[str]) -> pd.DataFrame:
    output = frame.copy()
    skill_sets = output["skills_norm_list"].apply(set)
    for skill in top_skills:
        column = "skill__" + re.sub(r"[^a-z0-9]+", "_", skill.lower()).strip("_")
        output[column] = skill_sets.apply(lambda values, skill=skill: int(skill in values))
    return output

## 3. Prepare Salary Target And Features

Target la `log_salary_monthly_vnd`. Cac cot salary goc va currency/period khong duoc dua vao feature matrix de tranh leakage.

In [ ]:
salary_model = prepare_salary_model_frame(salary_raw)
salary_model = salary_model.dropna(subset=["log_salary_monthly_vnd"]).copy()

assert len(salary_model) > 0, "model frame must have usable salary target rows"
assert salary_model["log_salary_monthly_vnd"].notna().all(), "target must be non-null"
assert salary_model["salary_monthly_vnd"].gt(0).all(), "monthly salary target must be positive"

print(f"Rows usable for salary model: {len(salary_model):,}")
display(
    salary_model[[
        "source",
        "title",
        "location_norm",
        "seniority",
        "work_mode",
        "experience_min",
        "skill_count",
        "salary_monthly_vnd_m",
        "log_salary_monthly_vnd",
    ]].head(10)
)

display(
    salary_model["salary_monthly_vnd_m"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
    .rename("monthly_salary_million_vnd")
    .to_frame()
)
display(pd.crosstab(salary_model["source"], salary_model["salary_currency"], dropna=False))

## 4. Train/Test Split And Top Skill Flags

Top skills duoc chon tu train set de tranh nhin truoc test distribution.

In [ ]:
train_base, test_base = train_test_split(
    salary_model,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=salary_model["source"],
)

train_skills = (
    train_base[["url", "skills_norm_list"]]
    .explode("skills_norm_list", ignore_index=True)
    .rename(columns={"skills_norm_list": "skill_norm"})
)
top_skills = (
    train_skills.loc[train_skills["skill_norm"].notna() & train_skills["skill_norm"].ne(""), "skill_norm"]
    .value_counts()
    .head(TOP_SKILL_COUNT)
    .index
    .tolist()
)

train_model = add_skill_flags(train_base, top_skills)
test_model = add_skill_flags(test_base, top_skills)

numeric_feature_columns = ["experience_min", "skill_count", "posted_age_days"]
categorical_feature_columns = ["source", "location_norm", "seniority", "work_mode", "employment_type"]
skill_feature_columns = [column for column in train_model.columns if column.startswith("skill__")]
feature_columns = numeric_feature_columns + skill_feature_columns + categorical_feature_columns

leakage_columns = {
    "salary_raw",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_monthly_vnd",
    "salary_monthly_vnd_m",
    "log_salary_monthly_vnd",
    "salary_currency",
    "salary_period",
    "salary_period_clean",
    "salary_label",
}

leaked_features = sorted(leakage_columns.intersection(feature_columns))
assert not leaked_features, f"Leakage columns included in features: {leaked_features}"

X_train = train_model[feature_columns].copy()
y_train = train_model["log_salary_monthly_vnd"]
X_test = test_model[feature_columns].copy()
y_test = test_model["log_salary_monthly_vnd"]

for column in categorical_feature_columns:
    X_train[column] = X_train[column].astype("object").where(X_train[column].notna(), np.nan)
    X_test[column] = X_test[column].astype("object").where(X_test[column].notna(), np.nan)

print(f"Train rows: {len(X_train):,}; Test rows: {len(X_test):,}")
print(f"Numeric features: {numeric_feature_columns}")
print(f"Categorical features: {categorical_feature_columns}")
print(f"Skill flags: {len(skill_feature_columns)}")
display(pd.Series(top_skills, name="top_train_skills").to_frame())

## 5. Fit LinearRegression

Pipeline chi fit `LinearRegression`. Numeric/categorical preprocessing dung de tao feature matrix hop le cho scikit-learn.

In [ ]:
numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_feature_columns + skill_feature_columns),
        ("categorical", categorical_transformer, categorical_feature_columns),
    ]
)

linear_regression_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression()),
    ]
)

linear_regression_model.fit(X_train, y_train)
pred_log = linear_regression_model.predict(X_test)

actual_salary_m = np.exp(y_test) / 1_000_000
pred_salary_m = np.exp(pred_log) / 1_000_000

metrics = pd.DataFrame(
    [
        {
            "model": "LinearRegression",
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            "feature_count_before_onehot": len(feature_columns),
            "log_mae": mean_absolute_error(y_test, pred_log),
            "log_rmse": float(np.sqrt(mean_squared_error(y_test, pred_log))),
            "log_r2": r2_score(y_test, pred_log),
            "salary_mae_million_vnd": mean_absolute_error(actual_salary_m, pred_salary_m),
        }
    ]
)

prediction_columns = [
    "url",
    "source",
    "title",
    "company",
    "location_norm",
    "seniority",
    "work_mode",
    "experience_min",
    "skill_count",
]
predictions = test_model[prediction_columns].copy()
predictions["actual_log_salary"] = y_test.to_numpy()
predictions["predicted_log_salary"] = pred_log
predictions["actual_salary_million_vnd"] = actual_salary_m.to_numpy()
predictions["predicted_salary_million_vnd"] = pred_salary_m
predictions["residual_log_salary"] = predictions["actual_log_salary"] - predictions["predicted_log_salary"]

metrics_path = MODELING_DIR / "salary_linear_regression_metrics.csv"
predictions_path = MODELING_DIR / "salary_linear_regression_predictions.csv"
metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
predictions.to_csv(predictions_path, index=False, encoding="utf-8-sig")

display(metrics)
display(predictions.head(10))
print(f"Wrote metrics: {metrics_path}")
print(f"Wrote predictions: {predictions_path}")

## 6. Salary Prediction Charts

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=predictions,
    x="actual_salary_million_vnd",
    y="predicted_salary_million_vnd",
    hue="source",
    alpha=0.75,
    ax=ax,
)
axis_min = min(predictions["actual_salary_million_vnd"].min(), predictions["predicted_salary_million_vnd"].min())
axis_max = max(predictions["actual_salary_million_vnd"].max(), predictions["predicted_salary_million_vnd"].max())
ax.plot([axis_min, axis_max], [axis_min, axis_max], color="crimson", linestyle="--", linewidth=1)
ax.set_title("LinearRegression: actual vs predicted salary")
ax.set_xlabel("Actual salary, million VND/month")
ax.set_ylabel("Predicted salary, million VND/month")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(
    data=predictions,
    x="predicted_log_salary",
    y="residual_log_salary",
    hue="source",
    alpha=0.75,
    ax=ax,
)
ax.axhline(0, color="crimson", linestyle="--", linewidth=1)
ax.set_title("Residuals on log salary target")
ax.set_xlabel("Predicted log salary")
ax.set_ylabel("Actual - predicted log salary")
plt.tight_layout()
plt.show()

## 7. Coefficient Inspection

Coefficient chi nen doc nhu tin hieu baseline. Categorical coefficients phu thuoc vao one-hot encoding va feature correlation.

In [ ]:
feature_names = linear_regression_model.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": linear_regression_model.named_steps["model"].coef_,
    }
)
coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
top_coefficients = coefficients.sort_values("abs_coefficient", ascending=False).head(30)

display(top_coefficients)

fig, ax = plt.subplots(figsize=(9, 8))
plot_coefficients = top_coefficients.sort_values("coefficient")
sns.barplot(data=plot_coefficients, x="coefficient", y="feature", ax=ax)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Top LinearRegression coefficients")
ax.set_xlabel("Coefficient on log salary")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Demand Trend Baseline

Day chi la baseline minh hoa. Dataset hien co khoang vai tuan posted date, nen khong nen xem day la forecast thi truong that.

In [ ]:
jobs = jobs_raw.copy()
jobs["posted_at_dt"] = pd.to_datetime(jobs["posted_at"], errors="coerce", utc=True, format="mixed")
jobs = jobs.dropna(subset=["posted_at_dt"]).copy()
posted_day = jobs["posted_at_dt"].dt.tz_convert(None).dt.normalize()
jobs["week_start"] = posted_day - pd.to_timedelta(posted_day.dt.weekday, unit="D")

weekly_demand = (
    jobs.groupby("week_start", as_index=False)
    .agg(job_count=("url", "nunique"))
    .sort_values("week_start")
    .reset_index(drop=True)
)
weekly_demand["week_index"] = np.arange(len(weekly_demand))

assert len(weekly_demand) >= 6, "need at least 6 weekly points for the demand baseline"

demand_model = LinearRegression()
demand_model.fit(weekly_demand[["week_index"]], weekly_demand["job_count"])
weekly_demand["predicted_job_count"] = demand_model.predict(weekly_demand[["week_index"]])
weekly_demand["row_type"] = "actual"

last_week = weekly_demand["week_start"].max()
future_weeks = pd.DataFrame(
    {
        "week_start": [last_week + pd.Timedelta(days=7), last_week + pd.Timedelta(days=14)],
        "week_index": [len(weekly_demand), len(weekly_demand) + 1],
        "job_count": [np.nan, np.nan],
        "row_type": ["forecast_baseline", "forecast_baseline"],
    }
)
future_weeks["predicted_job_count"] = demand_model.predict(future_weeks[["week_index"]])

demand_baseline = pd.concat([weekly_demand, future_weeks], ignore_index=True)
demand_baseline["predicted_job_count"] = demand_baseline["predicted_job_count"].clip(lower=0)
demand_baseline["note"] = "Exploratory LinearRegression baseline; short history, not a production forecast"

demand_path = MODELING_DIR / "demand_linear_regression_baseline.csv"
demand_baseline.to_csv(demand_path, index=False, encoding="utf-8-sig")

display(demand_baseline)
print(f"Demand weeks: {len(weekly_demand):,}")
print(f"Demand baseline slope: {demand_model.coef_[0]:.2f} jobs/week")
print(f"Wrote demand baseline: {demand_path}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
actual_rows = demand_baseline[demand_baseline["row_type"].eq("actual")]
forecast_rows = demand_baseline[demand_baseline["row_type"].eq("forecast_baseline")]

sns.scatterplot(data=actual_rows, x="week_start", y="job_count", s=80, label="Actual weekly jobs", ax=ax)
sns.lineplot(data=demand_baseline, x="week_start", y="predicted_job_count", color="crimson", marker="o", label="LinearRegression baseline", ax=ax)
if len(forecast_rows):
    sns.scatterplot(data=forecast_rows, x="week_start", y="predicted_job_count", color="crimson", marker="X", s=110, label="Next 2 weeks baseline", ax=ax)

ax.set_title("Weekly demand trend baseline")
ax.set_xlabel("Week start")
ax.set_ylabel("Unique job postings")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 9. Takeaways

- Salary model la phan chinh: `LinearRegression` predict `log_salary_monthly_vnd`.
- Demand trend chi la minh hoa voi history ngan; can nhieu tuan/thang hon truoc khi ket luan forecast.
- Feature matrix da assert khong chua salary leakage columns.